# 🎙️ VieNeu-TTS to OmniVoice Audio Reference & Fine-Tuning Pipeline
## 🇻🇳 Xuất 100% Voice Audio Presets Tiếng Việt & Huấn Luyện Voice Cloning

Notebook này phục vụ 3 mục tiêu chính:
1. **Render Reference Audio:** Dùng VieNeu-TTS v3 Turbo để render toàn bộ 20 giọng mẫu (Bắc, Trung, Nam) thành các file WAV chuẩn (48kHz, 16-bit PCM) kèm câu thoại mẫu.
2. **Zero-Shot Voice Cloning:** Nạp các file WAV vừa render vào **OmniVoice / F5-TTS** để nói tiếng Anh, tiếng Nhật, tiếng Pháp... bằng chất giọng người Việt.
3. **Fine-Tuning / Training:** Hướng dẫn chuẩn bị dataset và fine-tune mô hình F5-TTS / OmniVoice với giọng nói Tiếng Việt.

### Yêu cầu phần cứng:
- Chọn **Runtime -> Change runtime type -> GPU (T4 / V100 / A100)**.

In [ ]:
# Step 1: Cài đặt thư viện VieNeu-TTS, PyTorch và F5-TTS / OmniVoice stack
!nvidia-smi
%pip install -q "torch==2.8.0" "torchaudio==2.8.0" --index-url https://download.pytorch.org/whl/cu128
%pip install -q --upgrade --force-reinstall --no-deps "torchvision==0.23.0" --index-url https://download.pytorch.org/whl/cu128
%pip install -q "transformers==4.57.6" "git+https://github.com/pnnbao97/VieNeu-TTS.git@f56ce97ffb37" "soundfile==0.13.1" "f5-tts" "librosa" "gradio"

In [ ]:
# Step 2: Render 20 Giọng VieNeu-TTS Turbo thành các file WAV Reference Audio chuẩn
import os
import json
from pathlib import Path
import soundfile as sf
from vieneu import Vieneu

OUTPUT_DIR = Path("/content/voice_clone_refs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading VieNeu-TTS v3 Turbo model on GPU...")
tts = Vieneu(mode="v3turbo", device="cuda", backend="pytorch", backbone_repo="pnnbao-ump/VieNeu-TTS-v3-Turbo")

# Kịch bản thoại chuẩn tương ứng từng phong cách
STANDARD_PROMPTS = {
    "tin_tuc": "Trên thực tế, các chuyên gia kinh tế đã dự báo sự tăng trưởng mạnh mẽ của thị trường trong giai đoạn tới.",
    "doc_truyen": "Đêm dần về khuya, không gian trở nên tĩnh lặng, chỉ còn lại tiếng gió khẽ lay nhẹ qua những tán cây ngoài hiên.",
    "tu_nhien": "Chào bạn, hôm nay thời tiết rất trong lành và mát mẻ, chúc bạn một ngày làm việc tràn đầy năng lượng và niềm vui.",
    "le_hoi": "Tết là dịp mọi người háo hức đón chào một năm mới với nhiều hy vọng, may mắn và hạnh phúc sum vầy.",
    "khoa_hoc": "Ví dụ hai, tính giá trị trung bình và phân tích phương sai của dãy dữ liệu với độ chính xác cao.",
    "dam_thoai": "Hôm nay trời đẹp ghê, tụi mình cùng ghé quán cà phê quen ngồi tán gẫu một chút nha."
}

# Danh sách 20 Preset VieNeu Turbo
VOICES_CONFIG = [
    ("Minh Đức", "male", "bac", "tin_tuc"),
    ("Phạm Tuyên", "male", "bac", "tu_nhien"),
    ("Thái Sơn", "male", "nam", "doc_truyen"),
    ("Xuân Vĩnh", "male", "nam", "tu_nhien"),
    ("Thanh Bình", "male", "bac", "doc_truyen"),
    ("Trúc Ly", "female", "bac", "tu_nhien"),
    ("Ngọc Linh", "female", "bac", "doc_truyen"),
    ("Đoan Trang", "female", "bac", "tu_nhien"),
    ("Mai Anh", "female", "bac", "tin_tuc"),
    ("Thục Đoan", "female", "nam", "doc_truyen"),
    ("Minh Triết", "male", "nam", "tin_tuc"),
    ("Thùy Dung", "female", "nam", "tin_tuc"),
    ("Quang Sơn", "male", "trung", "tu_nhien"),
    ("Ngọc Trân", "female", "trung", "tu_nhien"),
    ("Mỹ Duyên", "female", "nam", "doc_truyen"),
    ("Quỳnh Anh", "female", "bac", "doc_truyen"),
    ("Đức Trí", "male", "nam", "doc_truyen"),
    ("Kim Thanh", "female", "nam", "doc_truyen"),
    ("Ngọc Huyền", "female", "bac", "tu_nhien"),
    ("Adam", "male", "nam", "tu_nhien")
]

manifest = []
for name, gender, region, style in VOICES_CONFIG:
    slug = name.lower().replace(" ", "_")
    g = "nam" if gender == "male" else "nu"
    wav_filename = f"vn_{slug}_{g}_{region}.wav"
    out_path = OUTPUT_DIR / wav_filename
    prompt_text = STANDARD_PROMPTS.get(style, STANDARD_PROMPTS["tu_nhien"])
    
    print(f"Generating sample for [{name}] ({gender}, {region}) -> {wav_filename}...")
    try:
        audio = tts.infer(prompt_text, voice=name)
        tts.save(audio, str(out_path))
        manifest.append({
            "voice": name,
            "file": wav_filename,
            "transcript": prompt_text,
            "gender": gender,
            "region": region,
            "style": style
        })
    except Exception as e:
        print(f"Error generating {name}: {e}")

# Lưu manifest kết quả
with open(OUTPUT_DIR / "manifest.json", "w", encoding="utf-8") as mf:
    json.dump(manifest, mf, ensure_ascii=False, indent=2)

!zip -r /content/vieneu_voice_clone_refs.zip /content/voice_clone_refs
print("Done! File ZIP sẵn sàng tải về tại /content/vieneu_voice_clone_refs.zip")

In [ ]:
# Step 3: Thử nghiệm Voice Cloning với F5-TTS / OmniVoice
# Ví dụ: Cho giọng Quang Sơn (Miền Trung) nói tiếng Anh và tiếng Việt đàm thoại
from f5_tts.api import F5TTS

f5 = F5TTS(model_type="F5-TTS", device="cuda")

ref_audio = "/content/voice_clone_refs/vn_quang_son_nam_trung.wav"
ref_text = STANDARD_PROMPTS["tu_nhien"]
gen_text = "Hello everyone! I am Quang Son from Central Vietnam, and I am speaking English through OmniVoice cloning technology!"

wav, sr, _ = f5.infer(
    ref_file=ref_audio,
    ref_text=ref_text,
    gen_text=gen_text,
    file_wave="/content/quang_son_english_cloned.wav"
)

print("Voice cloning completed: /content/quang_son_english_cloned.wav")

## 🚀 Fine-Tuning F5-TTS / OmniVoice với Dataset Tiếng Việt
Nếu bạn muốn huấn luyện (fine-tune) mô hình để phát âm Tiếng Việt tự nhiên hơn nữa hoặc thêm giọng riêng của bạn, chạy các bước bên dưới.

In [ ]:
# Step 4: Cấu trúc Dataset cho Huấn luyện F5-TTS / OmniVoice
"""
Cấu trúc thư mục dataset chuẩn:
dataset/
  ├── wavs/
  │   ├── audio_001.wav
  │   └── audio_002.wav
  └── metadata.csv (Format: audio_name|transcript|speaker_id)
"""
print("Fine-tuning command template:")
print("f5-tts_train --config_path /path/to/f5_train_config.yaml")